# 🔬 03 — Sanity Check (Interactive)

Run the 5 architecture validation tests interactively and visualize the overfit curve.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from model.cslt_model import CSLTModel
from training.loss import CSLTLoss
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

## 1. Create Model & Fake Data

In [ ]:
# Small model for fast testing
model = CSLTModel(input_dim=411, vocab_size=500, d_model=128, nhead=4,
                  num_encoder_layers=2, num_decoder_layers=2, dim_feedforward=256)

# Random data
B, T, D, S, V = 4, 30, 411, 20, 500
keypoints = torch.randn(B, T, D)
targets = torch.zeros(B, S, dtype=torch.long)
for i in range(B):
    n = torch.randint(5, S-2, (1,)).item()
    targets[i, 0] = 1  # <SOS>
    targets[i, 1:n+1] = torch.randint(4, V, (n,))
    targets[i, n+1] = 2  # <EOS>

model.count_parameters()

## 2. Test Forward Pass

In [ ]:
logits = model(keypoints, targets)
print(f'Input keypoints : {keypoints.shape}')
print(f'Input targets   : {targets.shape}')
print(f'Output logits   : {logits.shape}')
print(f'Expected        : ({B}, {S-1}, {V})')
assert logits.shape == (B, S-1, V)
assert not torch.isnan(logits).any()
print('\n✅ Forward pass OK!')

## 3. Overfit Test with Loss Curve

In [ ]:
criterion = CSLTLoss(vocab_size=V, pad_idx=0, label_smoothing=0.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
losses = []

model.train()
for step in range(100):
    logits = model(keypoints, targets)
    loss = criterion(logits, targets[:, 1:])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 20 == 0:
        print(f'  Step {step:3d} | Loss: {loss.item():.4f}')

print(f'\nInitial: {losses[0]:.4f} → Final: {losses[-1]:.4f}')
print(f'Reduction: {(1 - losses[-1]/losses[0])*100:.1f}%')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color='#E91E63', lw=2)
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.set_title('Overfit Test — Loss Should Decrease to ~0', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('\n✅ Model can learn!')

## 4. Freeze Test

In [ ]:
model2 = CSLTModel(input_dim=411, vocab_size=500, d_model=128, nhead=4,
                   num_encoder_layers=2, num_decoder_layers=2)
model2.freeze_encoder()

enc_before = {n: p.clone() for n, p in model2.encoder.named_parameters()}
opt2 = optim.Adam(filter(lambda p: p.requires_grad, model2.parameters()), lr=1e-3)
logits = model2(keypoints, targets)
loss = criterion(logits, targets[:, 1:])
opt2.zero_grad(); loss.backward(); opt2.step()

enc_changed = any(not torch.equal(p, enc_before[n]) for n, p in model2.encoder.named_parameters())
print(f'Encoder changed: {enc_changed} (expected: False)')
print('✅ Freeze test passed!' if not enc_changed else '❌ Freeze test FAILED')

## 5. Greedy Decoding Test

In [ ]:
model.eval()
with torch.no_grad():
    generated = model.translate(keypoints, max_len=15)

print(f'Generated shape: {generated.shape}')
print(f'Generated[0]: {generated[0].tolist()}')
print(f'Starts with <SOS>: {generated[0,0].item() == 1}')
print('\n✅ Greedy decoding works!')
print('\n🎉 ALL SANITY CHECKS PASSED — Architecture is valid!')